# 05 · Build the project tables
This notebook joins the three cleaned datasets into the two tables the analysis uses:

| Table | One row per | Used for |
|---|---|---|
| `panel_zip_daily.parquet` | ZIP × day × complaint type (zeros included) + that day's weather + the ZIP's housing stock | Weather effects and ZIP sensitivities |
| `building_heat_season.parquet` | Residential building × full heating season + heat complaints, apartments and year built | Building-age model |

The functions live in `src/build_features.py`.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
from src import config, preprocess as P, build_features as BF, analysis as A, plots as PL

pd.set_option("display.float_format", "{:,.2f}".format)
complaints = pd.read_parquet(config.CLEAN_311_FILE)
weather = pd.read_parquet(config.CLEAN_WEATHER_FILE)
pluto = pd.read_parquet(config.CLEAN_PLUTO_FILE)
print(f"311: {len(complaints):,} | weather: {len(weather)} days | PLUTO: {len(pluto):,} lots")

## 1. Enrich 311 with PLUTO: fill missing ZIPs via the building ID (BBL)

In [ ]:
rep = P.CleaningReport("311-enrichment")
complaints = P.impute_zip_from_pluto(complaints, pluto, rep)
complaints.to_parquet(config.CLEAN_311_FILE, index=False)
pd.DataFrame(rep.steps)

## 2. Housing profile per ZIP (exposure denominators)

In [ ]:
profile = BF.zip_housing_profile(pluto)
print(f"{len(profile)} ZIPs with residential lots")
profile.sort_values("res_units", ascending=False).head(10)

## 3. The ZIP × day panel
- **Complete grid:** days with zero complaints are included, which count models need.
- **Filter:** ZIPs with fewer than 500 apartments are excluded (mostly commercial or parkland ZIPs).

In [ ]:
panel = BF.build_zip_daily_panel(complaints, weather, pluto)
panel.to_parquet(config.PANEL_ZIP_DAILY_FILE, index=False)
print(f"{len(panel):,} rows | {panel.incident_zip.nunique()} ZIPs | {panel.date.nunique()} days | {panel.complaint_type.nunique()} types")
panel.head()

In [ ]:
# Sanity check: panel totals match the cleaned complaints (minus ZIPs excluded or missing)
in_panel = complaints[complaints.incident_zip.isin(panel.incident_zip.unique())]
check = pd.DataFrame({"panel": panel.groupby("complaint_type", observed=True)["n"].sum(),
                      "complaints_in_kept_zips": in_panel.groupby("complaint_type", observed=True).size(),
                      "all_complaints": complaints.groupby("complaint_type", observed=True).size()})
check.assign(coverage=check.panel / check.all_complaints)

## 4. The building × heat-season table

In [ ]:
print("Full heat seasons in window:", BF.full_heat_seasons())
buildings = BF.build_building_heat_table(complaints, pluto)
buildings.to_parquet(config.BUILDING_HEAT_FILE, index=False)
buildings.describe(include="all").T.head(12)

## 5. First look: the effect we will model

In [ ]:
PL.heat_vs_temperature(A.citywide_daily(panel, A.HEAT_SPEC));

In [ ]:
A.era_summary(buildings)

The tables are ready. Notebook 06 asks:
- How big are the weather effects?
- Do they vary by ZIP?
- Are neighbors alike?
- Does building age matter?

→ Next: **06_weather_sensitivity_analysis.ipynb**